# SOTA Classic BERT Variants — Baseline Evaluation

## Purpose

This notebook evaluates **standard state-of-the-art (SOTA) encoder-based transformer models** on downstream **text classification tasks**.
The results serve as a **static, full-depth baseline** for comparison against adaptive inference and early-exit strategies explored in subsequent notebooks.

All models evaluated here perform **fixed-depth inference**, meaning that every input is processed through **all transformer layers**, regardless of input difficulty.

--- 
    
## Evaluation Metrics

For each model, the following metrics are reported:

- **Accuracy**: classification performance on the evaluation set
- **Average inference latency**: wall-clock time per input sample
- **Fixed computational depth**: full transformer depth for all inputs

Latency is measured using **single-sample inference** to reflect realistic deployment conditions.

---

## Role in the Overall Project

This notebook provides the **static baseline** against which the following are compared:

- Zero-shot early-exit GPT-2 inference
- Fine-tuned early-exit GPT-2 models
- MLP-based learned exit strategies
- Shallow GPT-2 specialist classifiers

The results here define the **accuracy–latency reference point** for evaluating adaptive inference effectiveness.

---


In [5]:
# Clone Github Repo

import sys
import os
import subprocess

REPO_URL = "https://github.com/komalniraula/adaptive-inference-llm"
REPO_NAME = "adaptive-inference-llm"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL], check=True)

PROJECT_ROOT = os.path.abspath(REPO_NAME)
sys.path.insert(0, PROJECT_ROOT)

Cloning repository...


Cloning into 'adaptive-inference-llm'...


In [6]:
# Import necessary libraries

import pandas as pd
import numpy as np

from transformers import pipeline
from transformers import AutoTokenizer, ModernBertForSequenceClassification

import torch
import torch.nn.functional as F

In [7]:
# Load dataset

import time

from evaluation.dataset_loaders.sst2 import load_sst2

dataset_loaders = [
    ("sst2", load_sst2, "classification"),
    ]

cached_datasets = {}
print("Loading datasets once...\n")

for name, loader, task in dataset_loaders:
    print(f"Loading {name}...")
    cached_datasets[name] = {
        "data": loader(task='test', fraction=1),
        "task": task
    }

for data in cached_datasets:
    print(f"Length of the {data} dataset: {len(cached_datasets[data]['data'])}")

print("\nAll datasets loaded.\n")

Loading datasets once...

Loading sst2...
Length of the sst2 dataset: 872

All datasets loaded.



## Build Models

In [ ]:
from transformers import AutoModelForSequenceClassification

class RoBERTaLargeBaselineClassifier:
    def __init__(self, model_name="siebert/sentiment-roberta-large-english"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load tokenizer + fine-tuned RoBERTa-large sentiment classifier
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        self.model.to(self.device)
        self.model.eval()

        # RoBERTa-large has 24 encoder layers
        self.num_layers = self.model.config.num_hidden_layers  # = 24

    @torch.no_grad()
    def classify(self, text, dataset_name=None):

        # Tokenize
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=False,
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass through full model
        outputs = self.model(**inputs)
        logits = outputs.logits  # shape [1, 2]

        # Softmax probabilities
        probs = F.softmax(logits, dim=-1)[0]
        conf, pred = torch.max(probs, dim=0)

        # Return: predicted label, #layers used, confidence
        return pred.item(), self.num_layers, conf.item()


In [ ]:
class DistilBERTBaselineClassifier:
    def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load tokenizer + fine-tuned DistilBERT sentiment classifier
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

        self.model.to(self.device)
        self.model.eval()

        # DistilBERT has 6 encoder layers
        self.num_layers = self.model.config.num_hidden_layers  # = 6

    @torch.no_grad()
    def classify(self, text, dataset_name=None):
        # Tokenize text
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=False
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass
        outputs = self.model(**inputs)
        logits = outputs.logits  # shape [1, 2]

        # Compute probabilities
        probs = F.softmax(logits, dim=-1)[0]
        conf, pred = torch.max(probs, dim=0)

        # Baseline → always uses full 6 layers
        return pred.item(), self.num_layers, conf.item()

## Evaluation

In [ ]:
# Extract (text, label) from any format
def extract_text_label(sample):
    if isinstance(sample, dict):
        if "text" in sample:
            return sample["text"], sample["label"]
        elif "sentence" in sample:
            return sample["sentence"], sample["label"]
        elif "input_text" in sample:
            return sample["input_text"], sample["label"]
        else:
            raise ValueError("Unknown dict format:", sample)

    if isinstance(sample, (tuple, list)):
        return sample[0], sample[1]

    raise ValueError("Unknown sample format:", sample)

# Evaluation
def evaluate_dataset(model, dataset, dataset_name):
    correct = 0
    total = 0
    layers_used = []
    total_tokens = 0

    start = time.time()

    for sample in dataset:
        text, label = extract_text_label(sample)

        pred, layer, conf = model.classify(text, dataset_name)

        correct += (pred == label)
        total += 1
        layers_used.append(layer)

        total_tokens += len(model.tokenizer(text)["input_ids"])

    end = time.time()
    latency = (end - start) / total

    return {
        "metric": "accuracy",
        "score": correct / total,
        "avg_latency_sec": latency,
        "tokens_per_sec": total_tokens / (end - start),
        "avg_layers_used": float(np.mean(layers_used)),
        "num_samples": total
    }

## Run different models

In [ ]:
results_table = []

print("Running BASELINE DISTILBERT finetuned on SST2")

baseline_model = DistilBERTBaselineClassifier()

# Print number of layers (for logging/reporting)
print("DistilBERT sst2 finetuned model layers:", baseline_model.num_layers)

for name, meta in cached_datasets.items():
    dataset = meta["data"]
    print(f"\nTesting BASELINE on {name}...")

    result = evaluate_dataset(baseline_model, dataset, name)
    print(name, result)

    results_table.append({
        "dataset": name,
        "threshold": None,
        "mode": "baseline",
        "model": "Distil-BERT sst finetuned",
        "metric": result["metric"],
        "score": float(result["score"]),
        "avg_latency_sec": float(result["avg_latency_sec"]),
        "tokens_per_sec": float(result["tokens_per_sec"]),
        "avg_layers_used": float(result["avg_layers_used"]),
        "num_samples": int(result["num_samples"]),
    })

Running BASELINE DISTILBERT finetuned on SST2
DistilBERT sst2 finetuned model layers: 6

Testing BASELINE on sst2...
sst2 {'metric': 'accuracy', 'score': 0.9105504587155964, 'avg_latency_sec': 0.004897614684673624, 'tokens_per_sec': 5138.009509901858, 'avg_layers_used': 6.0, 'num_samples': 872}


In [ ]:
print("Running BASELINE DISTILBERT uncased")

baseline_model_distil_uncased = DistilBERTBaselineClassifier(model_name="distilbert/distilbert-base-uncased")

# Print number of layers (for logging/reporting)
print("DistilBERT uncased model layers:", baseline_model_distil_uncased.num_layers)

for name, meta in cached_datasets.items():
    dataset = meta["data"]
    print(f"\nTesting BASELINE on {name}...")

    result = evaluate_dataset(baseline_model, dataset, name)
    print(name, result)

    results_table.append({
        "dataset": name,
        "threshold": None,
        "mode": "baseline",
        "model": "Distil-BERT uncased",
        "metric": result["metric"],
        "score": float(result["score"]),
        "avg_latency_sec": float(result["avg_latency_sec"]),
        "tokens_per_sec": float(result["tokens_per_sec"]),
        "avg_layers_used": float(result["avg_layers_used"]),
        "num_samples": int(result["num_samples"]),
    })

Running BASELINE DISTILBERT uncased


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT uncased model layers: 6

Testing BASELINE on sst2...
sst2 {'metric': 'accuracy', 'score': 0.9105504587155964, 'avg_latency_sec': 0.004632979904839752, 'tokens_per_sec': 5431.491468245093, 'avg_layers_used': 6.0, 'num_samples': 872}


In [ ]:
print("Running BASELINE ROBERTA")

baseline_model_roberta = RoBERTaLargeBaselineClassifier()

print("RoBERTa Large model layers:", baseline_model_roberta.num_layers)

for name, meta in cached_datasets.items():
    dataset = meta["data"]
    print(f"\nTesting BASELINE on {name}...")

    result = evaluate_dataset(baseline_model_roberta, dataset, name)
    print(name, result)

    results_table.append({
        "dataset": name,
        "threshold": None,
        "mode": "baseline",
        "model": "ROBERTA-Large",
        "metric": result["metric"],
        "score": float(result["score"]),
        "avg_latency_sec": float(result["avg_latency_sec"]),
        "tokens_per_sec": float(result["tokens_per_sec"]),
        "avg_layers_used": float(result["avg_layers_used"]),
        "num_samples": int(result["num_samples"]),
    })

Running BASELINE ROBERTA
RoBERTa Large model layers: 24

Testing BASELINE on sst2...
sst2 {'metric': 'accuracy', 'score': 0.9243119266055045, 'avg_latency_sec': 0.015610218321511505, 'tokens_per_sec': 1666.1633922239846, 'avg_layers_used': 24.0, 'num_samples': 872}


In [ ]:
# Create df
df=pd.DataFrame(results_table)
df_sorted = (
    df.groupby("dataset", group_keys=True)
      .apply(lambda g: g.sort_values("score", ascending=False))
      .reset_index(drop=True)
)
df_sorted

/tmp/ipython-input-126550988.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("score", ascending=False))


,dataset,threshold,mode,model,metric,score,avg_latency_sec,tokens_per_sec,avg_layers_used,num_samples
0,sst2,None,baseline,ROBERTA-Large,accuracy,0.924312,0.015610,1666.163392,24.0,872
1,sst2,None,baseline,Distil-BERT sst finetuned,accuracy,0.910550,0.004898,5138.009510,6.0,872
2,sst2,None,baseline,Distil-BERT uncased,accuracy,0.910550,0.004633,5431.491468,6.0,872
